# LeetCode 1885 — Count Pairs in Two Arrays bằng Python

Link đề: https://leetcode.com/problems/count-pairs-in-two-arrays/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Binary Search, Sorting

> **Bài Premium.** Đề bài trong notebook được viết lại theo hiểu biết về bài (nguồn: bản mô tả của doocs/leetcode). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc trên LeetCode.

Skeleton LeetCode:

```python
class Solution:
    def countPairs(self, nums1: List[int], nums2: List[int]) -> int:
        ...
```

Lộ trình:
1. **Vét cạn mọi cặp** `i < j` (`O(n²)`).
2. **Biến đổi đại số**: `nums1[i] + nums1[j] > nums2[i] + nums2[j]` ⇔ `d[i] + d[j] > 0` với `d = nums1 − nums2` ⇒ bài toán đếm cặp trên **một** mảng.
3. **Sắp xếp + tìm kiếm nhị phân** (`bisect`) — `O(n log n)`.
4. **Sắp xếp + hai con trỏ** — `O(n log n)` (phần đếm `O(n)`).
5. **Cây Fenwick (BIT)** đếm trực tuyến theo thứ tự chỉ số, không cần sắp — `O(n log V)`.

## 1. Mục tiêu học

1. Dùng **biến đổi đại số** để gộp hai mảng thành một mảng hiệu `d`, đưa điều kiện về `d[i] + d[j] > 0` — "tách biến" theo chỉ số.
2. Hiểu vì sao điều kiện `i < j` **không cản trở** việc sắp xếp: đếm **cặp không thứ tự** không phụ thuộc thứ tự phần tử.
3. Đếm cặp có tổng vượt ngưỡng bằng **`bisect`** và bằng **hai con trỏ** trên mảng đã sắp.
4. Cài **Fenwick tree** để đếm "số phần tử trước đó lớn hơn `−d[j]`" — kỹ thuật dùng khi không được sắp xếp (ví dụ phải giữ thứ tự chỉ số).
5. Kiểm chứng bằng vét cạn trên input ngẫu nhiên, chú ý trường hợp `d[i] + d[j] = 0` (không tính) và số âm.

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên `nums1`, `nums2` cùng độ dài `n`. Đếm số cặp chỉ số `(i, j)` với `i < j` và

`nums1[i] + nums1[j] > nums2[i] + nums2[j]`.

**Ví dụ 1:** `nums1 = [2,1,2,1]`, `nums2 = [1,2,1,2]` → `1`. Chỉ cặp `(0, 2)`: `2 + 2 > 1 + 1`.

**Ví dụ 2:** `nums1 = [1,10,6,2]`, `nums2 = [1,4,1,5]` → `5`. Các cặp `(0,1), (0,2), (1,2), (1,3), (2,3)`.

**Ràng buộc:**
- `n == nums1.length == nums2.length`, `1 ≤ n ≤ 10⁵`
- `1 ≤ nums1[i], nums2[i] ≤ 10⁵`

In [1]:
import random
import time
from bisect import bisect_left, bisect_right
from typing import List

examples = [
    (([2, 1, 2, 1], [1, 2, 1, 2]), 1),
    (([1, 10, 6, 2], [1, 4, 1, 5]), 5),
]
for (a, b), want in examples:
    print(f"nums1={a}, nums2={b} -> kỳ vọng {want}")

nums1=[2, 1, 2, 1], nums2=[1, 2, 1, 2] -> kỳ vọng 1
nums1=[1, 10, 6, 2], nums2=[1, 4, 1, 5] -> kỳ vọng 5


## 3. Kiến thức nền

### 3.1 Mảng hiệu

`nums1[i] + nums1[j] > nums2[i] + nums2[j]` ⇔ `(nums1[i] − nums2[i]) + (nums1[j] − nums2[j]) > 0`. Đặt `d[i] = nums1[i] − nums2[i]` (có thể **âm**).

In [2]:
a, b = [1, 10, 6, 2], [1, 4, 1, 5]
d = [x - y for x, y in zip(a, b)]
print("d =", d)
print("cặp thỏa:", [(i, j) for i in range(4) for j in range(i + 1, 4) if d[i] + d[j] > 0])

d = [0, 6, 5, -3]
cặp thỏa: [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3)]


### 3.2 Đếm cặp không thứ tự không phụ thuộc thứ tự mảng

Số cặp `{i, j}` (`i ≠ j`) có `d[i] + d[j] > 0` là một tính chất của **đa tập** `d`. Đếm cặp `i < j` = đếm cặp không thứ tự ⇒ có thể **sắp xếp** `d` tùy ý.

In [3]:
def count_brute_d(d):
    return sum(1 for i in range(len(d)) for j in range(i + 1, len(d)) if d[i] + d[j] > 0)

d = [0, 6, 5, -3]
print(count_brute_d(d), count_brute_d(sorted(d)), count_brute_d(d[::-1]), "(như nhau)")

5 5 5 (như nhau)


### 3.3 `bisect_right` — đếm phần tử `≤ v` trong mảng đã sắp

`bisect_right(a, v)` = số phần tử `≤ v` ⇒ số phần tử `> v` là `len(a) − bisect_right(a, v)`. Có thể giới hạn đoạn tìm bằng tham số `lo`.

In [4]:
s = [-3, 0, 5, 6]
print(bisect_right(s, 0), "phần tử <= 0;", len(s) - bisect_right(s, -2), "phần tử > -2")
print("trong đoạn s[2:] có", len(s) - bisect_right(s, 3, lo=2), "phần tử > 3")

2 phần tử <= 0; 3 phần tử > -2
trong đoạn s[2:] có 2 phần tử > 3


### 3.4 Fenwick tree (Binary Indexed Tree)

Lưu tần suất trên miền giá trị `1..M`; `add(pos)` và `prefix(pos)` (số phần tử có vị trí `≤ pos`) đều `O(log M)`. Dùng để đếm "trong các phần tử đã thêm, có bao nhiêu phần tử lớn hơn `v`".

In [5]:
class BIT:
    def __init__(self, size):
        self.n = size; self.t = [0] * (size + 1)
    def add(self, i, v=1):
        while i <= self.n:
            self.t[i] += v; i += i & -i
    def prefix(self, i):
        s = 0
        while i > 0:
            s += self.t[i]; i -= i & -i
        return s

bit = BIT(10)
for pos in [3, 3, 7]:
    bit.add(pos)
print("số phần tử <= 5:", bit.prefix(5), "| <= 7:", bit.prefix(7))

số phần tử <= 5: 2 | <= 7: 3


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2: `d = [0, 6, 5, −3]`. Sắp: `[−3, 0, 5, 6]`.

Với mỗi phần tử `s[i]` (theo thứ tự đã sắp), đếm số phần tử **đứng sau** nó có `s[j] > −s[i]`:

| i | s[i] | cần s[j] > | các s[j] (j > i) | số thỏa |
|---|---|---|---|---|
| 0 | −3 | 3 | 0, 5, 6 | 2 |
| 1 | 0 | 0 | 5, 6 | 2 |
| 2 | 5 | −5 | 6 | 1 |
| 3 | 6 | — | — | 0 |

Tổng `5`. ✓

Vì mảng đã sắp, các `s[j]` thỏa điều kiện là một **đoạn cuối** ⇒ tìm bằng nhị phân. Hơn nữa, khi `i` tăng, ngưỡng `−s[i]` **giảm** ⇒ biên của đoạn chỉ dịch sang trái ⇒ hai con trỏ.

## 5. Cách 1 — Brute Force / baseline: mọi cặp `i < j`

In [6]:
def count_pairs_brute(nums1, nums2):
    n = len(nums1)
    cnt = 0
    for i in range(n):
        for j in range(i + 1, n):
            if nums1[i] + nums1[j] > nums2[i] + nums2[j]:
                cnt += 1
    return cnt

for (a, b), want in examples:
    print(count_pairs_brute(a, b), "kỳ vọng", want)

1 kỳ vọng 1
5 kỳ vọng 5


**Độ phức tạp:** Time `O(n²)` — `n = 10⁵` cho `5·10⁹` cặp. Space `O(1)`.

**Nút thắt:** với mỗi `j`, ta hỏi "có bao nhiêu `i` thỏa `d[i] > −d[j]`" — một câu hỏi **đếm theo ngưỡng** mà cấu trúc có thứ tự (mảng đã sắp, BIT) trả lời nhanh.

In [7]:
for n in [1000, 3000]:
    a = [random.randint(1, 10**5) for _ in range(n)]; b = [random.randint(1, 10**5) for _ in range(n)]
    t0 = time.perf_counter(); count_pairs_brute(a, b)
    print(f"n={n}: {time.perf_counter() - t0:.3f}s")

n=1000: 0.029s


n=3000: 0.232s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tách biến):** `d[i] + d[j] > 0` chỉ phụ thuộc **giá trị** `d`, không phụ thuộc chỉ số ⇒ sắp `d` tăng dần thành `s`.

**Insight 2 (đơn điệu):** với `s` tăng dần, cặp `(i, j)`, `i < j` thỏa ⇔ `s[j] > −s[i]`. Với `i` cố định, tập `j` thỏa là đoạn `[p(i), n − 1]` với `p(i) = bisect_right(s, −s[i], lo = i + 1)`.

**Insight 3 (hai con trỏ):** đặt `l = 0`, `r = n − 1`. Nếu `s[l] + s[r] > 0` thì **mọi** `l' ∈ [l, r − 1]` cũng có `s[l'] + s[r] > 0` (vì `s[l'] ≥ s[l]`) ⇒ cộng `r − l` cặp, rồi `r −= 1`. Ngược lại `s[l]` quá nhỏ để ghép với bất kỳ phần tử nào còn lại ⇒ `l += 1`.

**Invariant (hai con trỏ):** mọi cặp có một đầu ngoài `[l, r]` đã được đếm đúng (chính xác một lần).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp + `bisect`

In [8]:
def count_pairs_bisect(nums1, nums2):
    s = sorted(x - y for x, y in zip(nums1, nums2))
    n = len(s)
    total = 0
    for i in range(n):
        p = bisect_right(s, -s[i], lo=i + 1)
        total += n - p
    return total

for (a, b), want in examples:
    print(count_pairs_bisect(a, b), "kỳ vọng", want)

1 kỳ vọng 1
5 kỳ vọng 5


Tham số `lo = i + 1` bảo đảm chỉ đếm `j > i` (mỗi cặp một lần, không ghép phần tử với chính nó).

**Độ phức tạp:** sắp `O(n log n)` + `n` lần `bisect` `O(log n)` ⇒ `O(n log n)`. Space `O(n)`.

### 7.2 Cách 3 — Sắp xếp + hai con trỏ

In [9]:
def count_pairs_two_pointers(nums1, nums2):
    s = sorted(x - y for x, y in zip(nums1, nums2))
    l, r = 0, len(s) - 1
    total = 0
    while l < r:
        if s[l] + s[r] > 0:
            total += r - l
            r -= 1
        else:
            l += 1
    return total

for (a, b), want in examples:
    print(count_pairs_two_pointers(a, b), "kỳ vọng", want)

1 kỳ vọng 1
5 kỳ vọng 5


Code nộp LeetCode của Cách 3: xem mục 9.

**Độ phức tạp:** `O(n log n)` cho sắp, đếm `O(n)`. Space `O(n)`.

### 7.3 Cách 4 — Fenwick tree theo thứ tự chỉ số

Duyệt `j = 0..n−1`: số `i < j` thỏa `d[i] > −d[j]` = (số phần tử đã thêm) − (số phần tử `≤ −d[j]`). Thêm `d[j]` vào BIT. Giá trị `d ∈ [−(10⁵ − 1), 10⁵ − 1]` ⇒ dịch chỉ số `pos = d + OFFSET`.

In [10]:
def count_pairs_bit(nums1, nums2, LIM=10**5):
    OFF = LIM                                 # d trong [-(LIM-1), LIM-1] -> pos trong [1, 2*LIM-1]
    bit = BIT(2 * LIM + 1)
    total = 0
    for j, (x, y) in enumerate(zip(nums1, nums2)):
        dj = x - y
        # số i < j có d[i] > -dj  =  j - (số d[i] <= -dj)
        total += j - bit.prefix(-dj + OFF)
        bit.add(dj + OFF)
    return total

for (a, b), want in examples:
    print(count_pairs_bit(a, b), "kỳ vọng", want)

1 kỳ vọng 1
5 kỳ vọng 5


**Độ phức tạp:** Time `O(n log V)` với `V = 2·10⁵`, Space `O(V)`. Chậm hơn Cách 3 trong Python, nhưng là lựa chọn duy nhất khi **không được** sắp xếp (ví dụ: điều kiện có thêm ràng buộc phụ thuộc chỉ số như `j − i ≤ k`, hoặc truy vấn trực tuyến).

### 7.4 Cách 5 — Đếm theo tần suất trên miền giá trị nhỏ (prefix count)

Vì `d` nằm trong miền `[−10⁵, 10⁵]`, có thể đếm tần suất rồi dùng tổng tiền tố trên **giá trị**: với mỗi giá trị `v`, số phần tử `> −v` đọc được trong `O(1)`. Đếm cặp có thứ tự rồi trừ các cặp "tự ghép" (`2v > 0`) và chia 2.

In [11]:
def count_pairs_freq(nums1, nums2, LIM=10**5):
    OFF = LIM
    size = 2 * LIM + 1
    cnt = [0] * size
    d = [x - y for x, y in zip(nums1, nums2)]
    for v in d:
        cnt[v + OFF] += 1
    suf = [0] * (size + 1)                   # suf[k] = số phần tử có chỉ số giá trị >= k
    for k in range(size - 1, -1, -1):
        suf[k] = suf[k + 1] + cnt[k]
    ordered = 0
    for v in d:
        k = -v + OFF + 1                     # cần giá trị > -v  ->  chỉ số >= -v + OFF + 1
        ordered += suf[k] if 0 <= k <= size else (len(d) if k < 0 else 0)
    self_pairs = sum(1 for v in d if 2 * v > 0)    # cặp (i, i) bị đếm khi d[i] > -d[i]
    return (ordered - self_pairs) // 2

for (a, b), want in examples:
    print(count_pairs_freq(a, b), "kỳ vọng", want)

1 kỳ vọng 1
5 kỳ vọng 5


**Độ phức tạp:** Time `O(n + V)`, Space `O(V)`. Mẹo "đếm cặp có thứ tự, trừ cặp tự ghép, chia 2" dùng lại được ở rất nhiều bài đếm cặp.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 2 (`s = [−3, 0, 5, 6]`):

| l | r | s[l] + s[r] | hành động | total |
|---|---|---|---|---|
| 0 | 3 | −3 + 6 = 3 > 0 | cộng `r − l = 3`, `r = 2` | 3 |
| 0 | 2 | −3 + 5 = 2 > 0 | cộng `2`, `r = 1` | 5 |
| 0 | 1 | −3 + 0 = −3 | `l = 1` | 5 |
| 1 | 1 | dừng | | **5** |

In [12]:
def trace_two_pointers(nums1, nums2):
    s = sorted(x - y for x, y in zip(nums1, nums2)); l, r = 0, len(s) - 1; total = 0
    print("d đã sắp:", s)
    while l < r:
        if s[l] + s[r] > 0:
            total += r - l
            print(f"l={l}, r={r}: {s[l]} + {s[r]} = {s[l] + s[r]} > 0 -> cộng {r - l} (mọi l' trong [{l}, {r - 1}] ghép với r), total={total}")
            r -= 1
        else:
            print(f"l={l}, r={r}: {s[l]} + {s[r]} = {s[l] + s[r]} <= 0 -> s[l] quá nhỏ, l += 1")
            l += 1
    print("kết quả:", total)
    return total

trace_two_pointers([1, 10, 6, 2], [1, 4, 1, 5])

d đã sắp: [-3, 0, 5, 6]
l=0, r=3: -3 + 6 = 3 > 0 -> cộng 3 (mọi l' trong [0, 2] ghép với r), total=3
l=0, r=2: -3 + 5 = 2 > 0 -> cộng 2 (mọi l' trong [0, 1] ghép với r), total=5
l=0, r=1: -3 + 0 = -3 <= 0 -> s[l] quá nhỏ, l += 1
kết quả: 5


5

In [13]:
def trace_bisect(nums1, nums2):
    s = sorted(x - y for x, y in zip(nums1, nums2)); n = len(s); total = 0
    for i in range(n):
        p = bisect_right(s, -s[i], lo=i + 1); total += n - p
        print(f"i={i}: s[i]={s[i]:3d}, cần s[j] > {-s[i]:3d}, j từ {p} -> +{n - p}, total={total}")
    return total

trace_bisect([2, 1, 2, 1], [1, 2, 1, 2])

i=0: s[i]= -1, cần s[j] >   1, j từ 4 -> +0, total=0
i=1: s[i]= -1, cần s[j] >   1, j từ 4 -> +0, total=0
i=2: s[i]=  1, cần s[j] >  -1, j từ 3 -> +1, total=1
i=3: s[i]=  1, cần s[j] >  -1, j từ 4 -> +0, total=1


1

In [14]:
def trace_bit(nums1, nums2):
    d = [x - y for x, y in zip(nums1, nums2)]; seen = []; total = 0
    for j, dj in enumerate(d):
        good = [v for v in seen if v > -dj]
        total += len(good)
        print(f"j={j}: d[j]={dj:3d}, các d[i] trước đó {seen} lớn hơn {-dj}: {good} -> +{len(good)}, total={total}")
        seen.append(dj)
    return total

trace_bit([1, 10, 6, 2], [1, 4, 1, 5])

j=0: d[j]=  0, các d[i] trước đó [] lớn hơn 0: [] -> +0, total=0
j=1: d[j]=  6, các d[i] trước đó [0] lớn hơn -6: [0] -> +1, total=1
j=2: d[j]=  5, các d[i] trước đó [0, 6] lớn hơn -5: [0, 6] -> +2, total=3
j=3: d[j]= -3, các d[i] trước đó [0, 6, 5] lớn hơn 3: [6, 5] -> +2, total=5


5

## 9. Code nộp LeetCode

In [15]:
class Solution:
    def countPairs(self, nums1: List[int], nums2: List[int]) -> int:
        s = sorted(x - y for x, y in zip(nums1, nums2))
        l, r = 0, len(s) - 1
        total = 0
        while l < r:
            if s[l] + s[r] > 0:
                total += r - l
                r -= 1
            else:
                l += 1
        return total

sol = Solution()
for (a, b), want in examples:
    got = sol.countPairs(a, b)
    print(got, "OK" if got == want else "SAI")

1 OK
5 OK


## 10. Vì sao thuật toán đúng

**Biến đổi.** `nums1[i] + nums1[j] > nums2[i] + nums2[j] ⇔ d[i] + d[j] > 0` (chuyển vế). Số cặp `i < j` thỏa bằng số tập `{i, j}` thỏa ⇒ bất biến khi hoán vị `d`.

**Hai con trỏ.** Xét `s` tăng dần, các chỉ số còn "chưa xử lý" là `[l, r]`; invariant: mọi cặp có ít nhất một đầu ngoài `[l, r]` đã được đếm đúng.
- Nếu `s[l] + s[r] > 0`: với mọi `l' ∈ [l, r−1]`, `s[l'] + s[r] ≥ s[l] + s[r] > 0` ⇒ đúng `r − l` cặp chứa `r` (với đầu kia trong `[l, r−1]`) đều thỏa. Đếm xong mọi cặp chứa `r` trong đoạn ⇒ loại `r`.
- Nếu `s[l] + s[r] ≤ 0`: với mọi `r' ∈ [l+1, r]`, `s[l] + s[r'] ≤ s[l] + s[r] ≤ 0` ⇒ không cặp nào chứa `l` thỏa ⇒ loại `l`.
Mỗi bước loại một chỉ số và giữ invariant ⇒ khi `l ≥ r`, mọi cặp đã đếm.

**Bisect.** Với `s` tăng, `{j > i : s[j] > −s[i]}` là đoạn cuối bắt đầu ở `bisect_right(s, −s[i], lo = i+1)`.

**BIT.** Duyệt theo chỉ số, khi xét `j` thì BIT chứa đúng `d[0..j−1]`; `j − prefix(−d[j])` = số `i < j` có `d[i] > −d[j]`.

**Đếm tần suất.** Số cặp có thứ tự `(i, j)` (cho phép `i = j`) với `d[i] + d[j] > 0` là `Σ_i #{j : d[j] > −d[i]}`; trừ các cặp `i = j` (khi `2d[i] > 0`) rồi chia 2 vì điều kiện đối xứng.

## 11. Phân tích Time / Space

Gọi `n ≤ 10⁵`, `V = 2·10⁵` (miền giá trị của `d`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn | `O(n²)` | `O(1)` | kiểm chứng |
| 2. Sort + bisect | `O(n log n)` | `O(n)` | dễ hiểu |
| 3. Sort + hai con trỏ | `O(n log n)` | `O(n)` | **nộp bài** |
| 4. Fenwick | `O(n log V)` | `O(V)` | khi không được sắp |
| 5. Tần suất + tiền tố | `O(n + V)` | `O(V)` | miền giá trị nhỏ |

Kết quả tối đa `n(n−1)/2 ≈ 5·10⁹` — vượt `int32`, cần `long` trong C++/Java.

In [16]:
random.seed(1)
n = 10**5
a = [random.randint(1, 10**5) for _ in range(n)]; b = [random.randint(1, 10**5) for _ in range(n)]
for name, fn in [("bisect", count_pairs_bisect), ("2ptr", count_pairs_two_pointers), ("bit", count_pairs_bit),
                 ("freq", count_pairs_freq), ("submit", Solution().countPairs)]:
    t0 = time.perf_counter(); r = fn(a, b)
    print(f"{name:6s}: {r}  {time.perf_counter() - t0:.3f}s")
print("số cặp tối đa:", n * (n - 1) // 2, "> 2^31 - 1 =", 2**31 - 1)

bisect: 2504568793  0.040s
2ptr  : 2504568793  0.028s


bit   : 2504568793  0.134s
freq  : 2504568793  0.038s


submit: 2504568793  0.028s
số cặp tối đa: 4999950000 > 2^31 - 1 = 2147483647


## 12. Bộ test

In [17]:
fns = [count_pairs_bisect, count_pairs_two_pointers, count_pairs_bit, count_pairs_freq,
       lambda a, b: Solution().countPairs(a, b)]

for (a, b), want in examples:
    assert count_pairs_brute(a, b) == want
    for fn in fns:
        assert fn(a, b) == want

edge = [
    (([1], [1]), 0),                         # n = 1
    (([5, 5], [5, 5]), 0),                   # d = 0, 0: tổng 0 không tính
    (([2, 1], [1, 2]), 0),                   # d = 1, -1: tổng 0
    (([3, 3, 3], [1, 1, 1]), 3),             # mọi cặp
    (([1, 1, 1], [3, 3, 3]), 0),             # không cặp nào
    (([100000, 1], [1, 100000]), 0),         # d cực trị: 99999 + (-99999) = 0
    (([100000, 2], [1, 100000]), 1),
]
for (a, b), want in edge:
    assert count_pairs_brute(a, b) == want, (a, b)
    for fn in fns:
        assert fn(a, b) == want, (fn, a, b)

random.seed(2024)
for _ in range(1500):
    n = random.randint(1, 30)
    hi = random.choice([3, 10, 10**5])
    a = [random.randint(1, hi) for _ in range(n)]; b = [random.randint(1, hi) for _ in range(n)]
    want = count_pairs_brute(a, b)
    for fn in fns:
        assert fn(a, b) == want, (fn, a, b)
print("ví dụ + edge + 1500 bộ ngẫu nhiên khớp vét cạn")

ví dụ + edge + 1500 bộ ngẫu nhiên khớp vét cạn


In [18]:
random.seed(5)
for _ in range(20):
    n = random.randint(500, 2000)
    a = [random.randint(1, 50) for _ in range(n)]; b = [random.randint(1, 50) for _ in range(n)]
    want = count_pairs_brute(a, b)
    assert all(fn(a, b) == want for fn in fns)
print("n tới 2000, nhiều giá trị d bằng nhau: khớp vét cạn")

n tới 2000, nhiều giá trị d bằng nhau: khớp vét cạn


Edge `([100000, 1], [1, 100000])`: `d = [99999, −99999]`, tổng `0` ⇒ **không** tính (bất đẳng thức nghiêm ngặt) — kiểm tra cả biên của mảng `OFFSET` trong BIT/tần suất.

## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `>=` thay vì `>` (đếm cả tổng bằng 0)

In [19]:
def wrong_ge(nums1, nums2):
    s = sorted(x - y for x, y in zip(nums1, nums2)); l, r = 0, len(s) - 1; total = 0
    while l < r:
        if s[l] + s[r] >= 0:                  # SAI: đề yêu cầu > (nghiêm ngặt)
            total += r - l; r -= 1
        else:
            l += 1
    return total

for (a, b), want in examples:
    print("sai =", wrong_ge(a, b), "| đúng =", want)

sai = 5 | đúng = 1
sai = 5 | đúng = 5


**Sửa:** `> 0`. Với `bisect`, tương ứng dùng `bisect_right` (không phải `bisect_left`).

### Lỗi 2 — Sắp **riêng** `nums1` và `nums2` (mất liên kết chỉ số)

In [20]:
def wrong_sort_separately(nums1, nums2):
    a, b = sorted(nums1), sorted(nums2)
    return count_pairs_brute(a, b)

for (a, b), want in examples:
    print("sai =", wrong_sort_separately(a, b), "| đúng =", want)

sai = 0 | đúng = 1
sai = 6 | đúng = 5


Sắp riêng hai mảng làm `nums1[i]` không còn đi cặp với `nums2[i]`. **Sửa:** tạo `d[i] = nums1[i] − nums2[i]` **trước**, rồi mới sắp `d`.

### Lỗi 3 — `bisect` không giới hạn `lo = i + 1` (đếm cả `j ≤ i`)

In [21]:
def wrong_no_lo(nums1, nums2):
    s = sorted(x - y for x, y in zip(nums1, nums2)); n = len(s)
    return sum(n - bisect_right(s, -s[i]) for i in range(n))     # SAI: đếm cả (i, i) và đếm mỗi cặp 2 lần

for (a, b), want in examples:
    print("sai =", wrong_no_lo(a, b), "| đúng =", want)

sai = 4 | đúng = 1
sai = 12 | đúng = 5


Không giới hạn `lo` ⇒ mỗi cặp bị đếm **hai lần** (từ cả hai đầu) và có thể tính cả cặp `(i, i)` khi `s[i] > 0`. **Sửa:** `lo = i + 1` (hoặc trừ tự ghép rồi chia 2 như Cách 5).

### Lỗi 4 — Hai con trỏ: khi `s[l] + s[r] > 0` lại chỉ cộng 1

In [22]:
def wrong_add_one(nums1, nums2):
    s = sorted(x - y for x, y in zip(nums1, nums2)); l, r = 0, len(s) - 1; total = 0
    while l < r:
        if s[l] + s[r] > 0:
            total += 1; r -= 1                # SAI: phải cộng r - l
        else:
            l += 1
    return total

for (a, b), want in examples:
    print("sai =", wrong_add_one(a, b), "| đúng =", want)

sai = 1 | đúng = 1
sai = 2 | đúng = 5


**Sửa:** `total += r − l` — mọi `l' ∈ [l, r−1]` đều ghép được với `r`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Điều kiện dạng `f(i) + f(j) > g(i) + g(j)` ⇒ **chuyển vế** thành `h(i) + h(j) > 0` với `h = f − g`.
- "Đếm cặp `i < j` thỏa điều kiện đối xứng theo giá trị" ⇒ được **sắp xếp** ⇒ hai con trỏ / bisect.
- Điều kiện phụ thuộc cả thứ tự chỉ số ⇒ Fenwick / merge sort (đếm nghịch thế).
- Đếm cặp có thứ tự − tự ghép, chia 2 = đếm cặp không thứ tự.
- Họ bài: 2563 *Count the Number of Fair Pairs*, 1679 *Max Number of K-Sum Pairs*, 2426 *Number of Pairs Satisfying Inequality* (cần BIT vì có ràng buộc chỉ số), 493 *Reverse Pairs*.

## 15. Biến thể của bài

1. **Ngưỡng tùy ý:** đếm `d[i] + d[j] > k` — hai con trỏ với `s[l] + s[r] > k`.
2. **Tổng trong đoạn `[lo, hi]`** (2563 Fair Pairs): `count(> lo − 1) − count(> hi)`.
3. **2426:** `nums1[i] − nums1[j] ≤ nums2[i] − nums2[j] + diff` với `i < j` ⇔ `d[i] ≤ d[j] + diff` — phụ thuộc thứ tự ⇒ BIT hoặc merge sort.
4. **Đếm bộ ba** `d[i] + d[j] + d[k] > 0`: sắp + với mỗi `i` dùng hai con trỏ ⇒ `O(n²)`.

In [23]:
def count_greater_than(d, k):
    s = sorted(d); l, r = 0, len(s) - 1; total = 0
    while l < r:
        if s[l] + s[r] > k:
            total += r - l; r -= 1
        else:
            l += 1
    return total

def fair_pairs(nums, lower, upper):
    """Biến thể 2 (LeetCode 2563): số cặp i<j với lower <= nums[i]+nums[j] <= upper."""
    return count_greater_than(nums, lower - 1) - count_greater_than(nums, upper)

def count_2426(nums1, nums2, diff):
    """Biến thể 3 (LeetCode 2426): i<j, d[i] <= d[j] + diff, dùng BIT theo thứ tự chỉ số."""
    d = [x - y for x, y in zip(nums1, nums2)]
    vals = sorted(set(d))
    bit = BIT(len(vals)); total = 0
    for v in d:
        total += bit.prefix(bisect_right(vals, v + diff))   # số d[i] (đã thêm) <= v + diff
        bit.add(bisect_left(vals, v) + 1)
    return total

assert fair_pairs([0, 1, 7, 4, 4, 5], 3, 6) == 6 and fair_pairs([1, 7, 9, 2, 5], 11, 11) == 1
assert count_2426([3, 2, 5], [2, 2, 1], 1) == 3 and count_2426([3, -1], [-2, 2], -1) == 0
random.seed(3)
for _ in range(300):
    n = random.randint(1, 12)
    a = [random.randint(-5, 5) for _ in range(n)]; b = [random.randint(-5, 5) for _ in range(n)]; df = random.randint(-3, 3)
    brute = sum(1 for i in range(n) for j in range(i + 1, n) if a[i] - a[j] <= b[i] - b[j] + df)
    assert count_2426(a, b, df) == brute
print("biến thể 2563 và 2426 khớp ví dụ / vét cạn")

biến thể 2563 và 2426 khớp ví dụ / vét cạn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi cặp | `O(n²)` | `O(1)` | kiểm chứng |
| 2. Sort + bisect | đoạn cuối thỏa | `O(n log n)` | `O(n)` | dễ viết |
| 3. Sort + hai con trỏ | cộng `r − l` | `O(n log n)` | `O(n)` | **nộp bài** |
| 4. Fenwick | đếm trực tuyến theo chỉ số | `O(n log V)` | `O(V)` | điều kiện phụ thuộc thứ tự |
| 5. Tần suất | tiền tố trên giá trị | `O(n + V)` | `O(V)` | giá trị nhỏ |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `countPairs([1,2,3], [3,2,1])` và `countPairs([4,4,4,4], [1,1,1,1])`?

**Bài 2 (tính tay):** `nums1 = [5, 1, 3, 2]`, `nums2 = [2, 4, 3, 1]`: tính `d`, sắp, chạy hai con trỏ bằng tay.

**Bài 3 (điền TODO):** hoàn thiện `pairs_list(nums1, nums2)` trả về **danh sách** các cặp `(i, j)` (chỉ số gốc, `i < j`) thỏa điều kiện — vẫn dùng sắp xếp để không xét cặp vô ích (đầu ra có thể tới `O(n²)`).

**Bài 4 (tự viết lại):** viết `count_at_least_k(nums1, nums2, k)` đếm cặp có `(nums1[i] + nums1[j]) − (nums2[i] + nums2[j]) ≥ k`.

**Bài 5 (mini challenge):** đếm **bộ ba** `i < j < l` với `d[i] + d[j] + d[l] > 0` trong `O(n²)`; kiểm chứng bằng vét cạn `O(n³)`. (Biến thể tự đặt.)

In [24]:
def pairs_list(nums1, nums2):
    # TODO: sắp chỉ số theo d; với mỗi vị trí trong thứ tự sắp, lấy đoạn cuối thỏa và ánh xạ về chỉ số gốc
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `d = [−2, 0, 2]`. (2) Cặp `(−2, 0)`: −2; `(−2, 2)`: 0 (không tính); `(0, 2)`: 2 ✓. (3) Đáp án `1`; mảng thứ hai `d = [3,3,3,3]` ⇒ `C(4,2) = 6`.

**Bài 2:** (1) `d = [3, −3, 0, 1]`, sắp `[−3, 0, 1, 3]`. (2) `l=0,r=3`: `0` không `> 0` ⇒ `l=1`; `l=1,r=3`: `3 > 0` ⇒ `+2`, `r=2`; `l=1,r=2`: `1 > 0` ⇒ `+1`. (3) Đáp án `3`.

**Bài 3:** (1) `order = sorted(range(n), key=lambda i: d[i])`. (2) Với mỗi vị trí `p` trong `order`, `q = bisect` trên mảng `d` đã sắp. (3) Cặp `(min(order[p], order[t]), max(...))` với `t ≥ q`.

**Bài 4:** (1) `≥ k` ⇔ `> k − 1` (số nguyên). (2) Dùng `count_greater_than(d, k − 1)`. (3) Thử `k = 1` khớp bài gốc.

**Bài 5:** (1) Sắp `s`. (2) Với mỗi `i`, đếm cặp `(j, l)` trong `s[i+1:]` có `s[j] + s[l] > −s[i]` bằng hai con trỏ. (3) Tổng `O(n²)`.

## 19. Đáp án tham khảo

In [25]:
# Bài 1
assert Solution().countPairs([1, 2, 3], [3, 2, 1]) == 1 and Solution().countPairs([4, 4, 4, 4], [1, 1, 1, 1]) == 6

# Bài 2
assert Solution().countPairs([5, 1, 3, 2], [2, 4, 3, 1]) == 3

# Bài 3
def pairs_list_ans(nums1, nums2):
    n = len(nums1); d = [x - y for x, y in zip(nums1, nums2)]
    order = sorted(range(n), key=lambda i: d[i]); s = [d[i] for i in order]; out = []
    for p in range(n):
        q = bisect_right(s, -s[p], lo=p + 1)
        for t in range(q, n):
            i, j = order[p], order[t]
            out.append((min(i, j), max(i, j)))
    return sorted(out)

# Bài 4
def count_at_least_k(nums1, nums2, k):
    return count_greater_than([x - y for x, y in zip(nums1, nums2)], k - 1)

random.seed(181)
for _ in range(500):
    n = random.randint(1, 12)
    a = [random.randint(1, 9) for _ in range(n)]; b = [random.randint(1, 9) for _ in range(n)]
    assert pairs_list_ans(a, b) == [(i, j) for i in range(n) for j in range(i + 1, n) if a[i] + a[j] > b[i] + b[j]]
    k = random.randint(-5, 5)
    assert count_at_least_k(a, b, k) == sum(1 for i in range(n) for j in range(i + 1, n) if a[i] + a[j] - b[i] - b[j] >= k)
print("Bài 1-4: OK; ví dụ 2 ->", pairs_list_ans([1, 10, 6, 2], [1, 4, 1, 5]))

Bài 1-4: OK; ví dụ 2 -> [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3)]


In [26]:
# Bài 5 — bộ ba
def count_triples(d):
    s = sorted(d); n = len(s); total = 0
    for i in range(n):
        l, r = i + 1, n - 1; target = -s[i]
        while l < r:
            if s[l] + s[r] > target:
                total += r - l; r -= 1
            else:
                l += 1
    return total

def count_triples_brute(d):
    n = len(d)
    return sum(1 for i in range(n) for j in range(i + 1, n) for l in range(j + 1, n) if d[i] + d[j] + d[l] > 0)

random.seed(182)
for _ in range(400):
    d = [random.randint(-6, 6) for _ in range(random.randint(0, 14))]
    assert count_triples(d) == count_triples_brute(d)
print("Bài 5: đếm bộ ba O(n^2) khớp vét cạn; ví dụ d = [0,6,5,-3] ->", count_triples([0, 6, 5, -3]))

Bài 5: đếm bộ ba O(n^2) khớp vét cạn; ví dụ d = [0,6,5,-3] -> 4


## 20. Template nhớ nhanh + Checklist

```python
# Đếm cặp i<j với a[i]+a[j] > b[i]+b[j]
s = sorted(x - y for x, y in zip(a, b))      # tách biến: d = a - b
l, r, total = 0, len(s) - 1, 0
while l < r:
    if s[l] + s[r] > 0:
        total += r - l                         # mọi l' in [l, r-1] ghép được với r
        r -= 1
    else:
        l += 1                                 # s[l] không ghép được với ai còn lại
return total
```

**Checklist trước khi submit:**
- [ ] Tạo `d = nums1 − nums2` **trước** khi sắp (không sắp riêng hai mảng).
- [ ] Bất đẳng thức **nghiêm ngặt** `> 0` (bisect: dùng `bisect_right`).
- [ ] Hai con trỏ cộng `r − l`, không cộng 1.
- [ ] Bisect giới hạn `lo = i + 1` để không đếm trùng.
- [ ] Kết quả có thể tới `~5·10⁹` (C++/Java: `long`).